In [ ]:
import sys
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.table import Table
import astropy
import sncosmo
import lcdata
import parsnip
from tqdm import tqdm

# Try importing ztfidr
try:
    import ztfidr
except ImportError:
    print("Warning: 'ztfidr' module not found. Ensure it is installed or in your python path.")

# --- CONFIGURATION ---
ZTFIDRPATH = os.getenv("ZTFIDRPATH", "./data/ztfcosmoidr/dr2/")
ZTFDATA = os.getenv("ZTFDATA", "./data/")
PARSNIP_DIR = os.getenv("PARSNIP_DIR", "./parsnip")

# Set environment variables for shell commands
os.environ['PARSNIP_DIR'] = PARSNIP_DIR
os.environ['ZTFIDRPATH'] = ZTFIDRPATH
os.environ['ZTFDATA'] = ZTFDATA

print(f"Using PARSNIP_DIR: {PARSNIP_DIR}")

# Create directories
os.makedirs(os.path.join(PARSNIP_DIR, 'figures'), exist_ok=True)
os.makedirs(os.path.join(PARSNIP_DIR, 'data'), exist_ok=True)
os.makedirs(os.path.join(PARSNIP_DIR, 'predictions'), exist_ok=True)

In [ ]:
# Access the ZTF sample
sample = ztfidr.get_sample()

# Create a small subset of N supernovas
N = 3000
sample_subset = sample.data[0:N]

# Convert to Astropy-table format for easier handling
sample_subset_astropy = Table.from_pandas(sample_subset)
print(f"Created subset with {len(sample_subset)} objects.")

In [ ]:
# Convert ZTF light-curves to lcdata format 

light_curves = []
print("Starting conversion of ZTF light curves...")

# Iterate through the subset

for i in tqdm(range(len(sample_subset)-1)):
    
    # 1. Extract metadata from the sample row
    row = sample_subset.iloc[i]
    
    # Reconstruct the Supernova ID
    # The original index is stored as a character array/list, so we join it into a string
    try:
        sn_id = "".join(sample_subset.index[i])
    except TypeError:
        sn_id = str(sample_subset.index[i])
        
    # 2. Standardize Classification (Type Ia vs. Unknown)
    raw_type = row.get('classification', 'Unknown')
    
    # Handle non-string types (NaNs) and normalize
    if not isinstance(raw_type, str):
        sn_type = 'Unknown'
    elif 'a' in raw_type:
        # Group all Type Ia variants (normal, strange, etc.) into 'SNIa'
        sn_type = 'SNIa'
    else:
        # Treat everything else as Unknown for this binary study
        sn_type = 'Unknown'

    # 3. Fetch the light curve data
    try:
        target = sample.get_target_lightcurve(sn_id)
        lc_data = target.get_lcdata() # Returns a pandas DataFrame/Series compatible object
    except Exception as e:
        print(f"Warning: Could not fetch data for {sn_id}. Skipping.")
        continue
    
    # 4. Map ZTF filters to PS1 equivalent (Required by ParSNIP)
    # We modify a copy of the filter column to avoid setting warnings
    bands = lc_data['filter'].copy()
    
    for idx, val in bands.items():
        if 'g' in val:
            bands[idx] = 'ps1::g'
        elif 'r' in val:
            bands[idx] = 'ps1::r'
        elif 'i' in val:
            bands[idx] = 'ps1::i'
        elif 'z' in val:
            bands[idx] = 'ps1::z'
        elif 'y' in val:
            bands[idx] = 'ps1::y'

    # 5. Create the Astropy Table
    # Construct the final table for this object
    new_lc = Table({
        'bandpass': bands,
        'flux': lc_data['flux'],
        'mjd': lc_data['mjd'],
        'fluxerr': lc_data['error']
    })

    # Add required metadata
    new_lc.meta = {
        'id': sn_id,
        'right_ascension': np.nan, # Placeholder if not in source
        'declination': np.nan,     # Placeholder if not in source
        'class': sn_type,
        'other_var': np.nan,
        'redshift': row.get('redshift', np.nan)
    }

    light_curves.append(new_lc)

print(f"Successfully formatted {len(light_curves)} light curves.")

In [ ]:
# Write the dataset in HDF5 format
ztf_subset_path = os.path.join(PARSNIP_DIR, 'data', 'dataset_ztf_subset.h5')
dataset_ztf.write_hdf5(ztf_subset_path, overwrite=True, append=False)
print(f"ZTF subset saved to: {ztf_subset_path}")

In [ ]:
# Generate predictions using the pre-trained ParSNIP model
!parsnip_predict "{PARSNIP_DIR}/predictions/parsnip_predictions_ztf_subset.h5" \
    "{PARSNIP_DIR}/models/parsnip_ps1.pt" \
    "{PARSNIP_DIR}/data/dataset_ztf_subset.h5" \
    --augments 100

In [ ]:
# Load the predictions
pred_file_ztf = os.path.join(PARSNIP_DIR, 'predictions', 'parsnip_predictions_ztf_subset.h5')
raw_predictions_ztf = Table.read(pred_file_ztf)
predictions_ztf = raw_predictions_ztf.copy()

# Train classifier and plot confusion matrix
classifier_ztf = parsnip.Classifier()
classifications_ztf = classifier_ztf.train(predictions_ztf, target_label='SNIa')

parsnip.plot_confusion_matrix(predictions_ztf, classifications_ztf, title='ZTF - ParSNIP')

# Save figure
fig_path_ztf = os.path.join(PARSNIP_DIR, 'figures', 'ztf_confusion_matrix.pdf')
plt.savefig(fig_path_ztf)
print(f"Confusion matrix saved to {fig_path_ztf}")

In [ ]:
### Visualize specific examples of misclassified vs. correctly classified objects

name1 = "ZTF17aadlxmv" #well identified sn1a
name2 = "ZTF18aaapivw" #well identified unknown object
name3='ZTF18aaadqua' #not well identified sn1a
name4='ZTF19aahsclk' #not well identified unknown

###get the light curves of the studied supernovae
sn1 = sample.get_target_lightcurve(name1)
sn2 = sample.get_target_lightcurve(name2)
sn3 = sample.get_target_lightcurve(name3)
sn4 = sample.get_target_lightcurve(name4)

In [ ]:
sn4.show()

In [ ]:
sn3.show()

In [ ]:
sn1.show()

In [ ]:
sn2.show()

In [ ]:
### Get the characteristics of the studied supernovae


names=[name1,name2,name3,name4]
for i in range (len(names)):
    sn=sample.get_target_lightcurve(names[i]) ### get the light curve of the studied supernova in the panda dataframe
    print(sn.salt2param)    ###get its characteristics
